# RAGRoute 재현 — Colab 실행 노트북

**실행 전 준비사항**
- 런타임: GPU (A100 권장)
- Colab Secrets에 `HF_TOKEN` 저장
- Cell 1의 `REPO_URL`을 본인 repo URL로 수정
- Google Drive `MyDrive/ragroute/benchmark/`에 `MIRAGE.json` + `question_order_MIRAGE_*.json` (5개) 업로드

**실행 순서**: Cell 1 → 2 → 3 → 4 → 5 → 6 → (7→8→9 선택)

In [ ]:
import os, sys, shutil, subprocess, torch
from google.colab import drive, userdata

# ⚠️ 여기만 수정
REPO_URL = 'https://github.com/ByoungjaeMin/ragroute_recreation.git'

assert torch.cuda.is_available(), 'GPU 없음: 런타임 > 런타임 유형 변경'
print(f'GPU: {torch.cuda.get_device_name(0)}')

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/ragroute'

REPO_PATH = '/content/ragroute_recreation'
if not os.path.isdir(REPO_PATH):
    subprocess.run(['git', 'clone', REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '--ff-only'])
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

def _link(local, sub):
    d = f'{DRIVE_ROOT}/{sub}'; os.makedirs(d, exist_ok=True)
    if os.path.islink(local): os.remove(local)
    elif os.path.isdir(local): shutil.rmtree(local)
    os.symlink(d, local)

for local, sub in [('data/embeddings','embeddings'), ('data/processed','processed'),
                   ('data/stats','stats'), ('data/raw','raw'),
                   ('checkpoints','checkpoints'), ('results','results')]:
    _link(local, sub)

os.makedirs('data/benchmark', exist_ok=True)
bdir = f'{DRIVE_ROOT}/benchmark'
for fn in ['MIRAGE.json']:
    shutil.copy2(f'{bdir}/{fn}', f'data/benchmark/{fn}')
for b in ['bioasq','medmcqa','medqa','mmlu-med','pubmedqa']:
    shutil.copy2(f'{bdir}/question_order_MIRAGE_{b}.json', f'data/question_order_MIRAGE_{b}.json')

print('✅ 환경 설정 완료')

In [ ]:
import os, subprocess, shutil
from huggingface_hub import snapshot_download

os.makedirs('data/raw/mirage', exist_ok=True)

# textbooks — HuggingFace (chunk/ → data/raw/mirage/textbooks/)
tb_dst = 'data/raw/mirage/textbooks'
if os.path.isdir(tb_dst) and os.listdir(tb_dst):
    print('[SKIP] textbooks')
else:
    print('[DOWN] textbooks ...')
    tmp = '/tmp/textbooks_hf'
    snapshot_download('MedRAG/textbooks', repo_type='dataset', local_dir=tmp,
                      allow_patterns=['chunk/*'], token=os.environ.get('HF_TOKEN'))
    shutil.copytree(f'{tmp}/chunk', tb_dst)
    print(f'  완료: {len(os.listdir(tb_dst))} files')

# statpearls — NCBI FTP + MedRAG statpearls.py
sp_dst = 'data/raw/mirage/statpearls'
if os.path.isdir(sp_dst) and os.listdir(sp_dst):
    print('[SKIP] statpearls')
else:
    medrag = '/tmp/MedRAG_toolkit'
    if not os.path.isdir(medrag):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/Teddy-XiongGZ/MedRAG.git', medrag], check=True)

    ncbi_tar = f'{medrag}/corpus/statpearls/statpearls_NBK430685.tar.gz'
    os.makedirs(os.path.dirname(ncbi_tar), exist_ok=True)
    if not os.path.exists(ncbi_tar):
        print('[DOWN] statpearls NCBI FTP (~1.7GB) ...')
        subprocess.run(['wget', '-q', '--show-progress', '-O', ncbi_tar,
                        'https://ftp.ncbi.nlm.nih.gov/pub/litarch/3d/12/statpearls_NBK430685.tar.gz'], check=True)

    nxml = f'{medrag}/corpus/statpearls/statpearls_NBK430685'
    if not os.path.isdir(nxml):
        subprocess.run(['tar', 'xzf', ncbi_tar, '-C', os.path.dirname(nxml)], check=True)

    chunk_out = f'{medrag}/corpus/statpearls/chunk'
    if not (os.path.isdir(chunk_out) and os.listdir(chunk_out)):
        print('[PROC] statpearls.py (~20분) ...')
        subprocess.run(['python3', 'src/data/statpearls.py'], cwd=medrag, check=True)

    shutil.copytree(chunk_out, sp_dst)
    print(f'  완료: {len(os.listdir(sp_dst))} files')

# Wikipedia 1M 스니펫 생성 (MMLU용)
SNIPPETS_PATH = 'data/raw/wikipedia_1m/snippets.jsonl'
if os.path.exists(SNIPPETS_PATH):
    import subprocess as _sp
    n = int(_sp.run(['wc','-l',SNIPPETS_PATH], capture_output=True, text=True).stdout.split()[0])
    print(f'[SKIP] Wikipedia snippets ({n:,}줄)')
else:
    print('[GEN] Wikipedia 1M 스니펫 생성 중 (수십 분 소요)...')
    from datasets import load_dataset
    os.makedirs('data/raw/wikipedia_1m', exist_ok=True)
    ds = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
    count = 0; target = 1_000_000
    with open(SNIPPETS_PATH, 'w') as f:
        for article in ds:
            words = article['text'].split()
            for i in range(0, len(words), 100):
                f.write(json.dumps({'title': article['title'], 'text': ' '.join(words[i:i+100])}) + '\n')
                count += 1
                if count >= target: break
            if count >= target: break
    print(f'  완료: {count:,} 스니펫')

print('✅ 데이터 준비 완료')

In [ ]:
# medrag: MedCPT-Article-Encoder (textbooks/statpearls) + MedCPT-Query-Encoder (query)
# A100 기준 ~30분
!python scripts/02_build_embeddings.py --dataset medrag
!python scripts/02_build_embeddings.py --dataset mmlu

In [ ]:
# medrag: IndexFlatL2 / mmlu: normalize_L2 + IndexFlatIP
!python scripts/03_build_index.py --dataset medrag
!python scripts/03_build_index.py --dataset mmlu

In [ ]:
# LABEL_K=15, medrag=L2 오름차순, mmlu=IP 내림차순, test=60%
!python scripts/04_generate_train_data.py --config experiments/mirage_top32.yaml
!python scripts/04_generate_train_data.py --config experiments/mmlu_top10.yaml

In [ ]:
# 150 epoch, CyclicLR(ep<115) → StepLR(ep≥115)
# medrag best=val_auc, wikipedia best=val_f1  |  A100 기준 각 5~15분
!python scripts/05_train_router.py --config experiments/mirage_top32.yaml
!python scripts/05_train_router.py --config experiments/mmlu_top10.yaml

In [ ]:
import pickle, torch, numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from src.config import INPUT_DIM
from src.router_model import CorpusRoutingNN

def _eval(dataset, proc_dir, ckpt):
    X = np.load(f'{proc_dir}/X_test.npy').astype(np.float32)
    y = np.load(f'{proc_dir}/y_test.npy')
    m = CorpusRoutingNN(INPUT_DIM[dataset])
    m.load_state_dict(torch.load(f'checkpoints/{ckpt}_model_best.pth', map_location='cpu', weights_only=True))
    m.eval()
    with open(f'checkpoints/{ckpt}_scaler.pkl','rb') as f: sc = pickle.load(f)
    X = sc.transform(X).astype(np.float32)
    with torch.no_grad(): p = torch.sigmoid(m(torch.tensor(X)).squeeze(1)).numpy()
    pred = (p >= 0.5).astype(int)
    return (accuracy_score(y,pred)*100, recall_score(y,pred,zero_division=0)*100,
            f1_score(y,pred,zero_division=0)*100,
            roc_auc_score(y,p)*100 if len(np.unique(y))>1 else float('nan'))

TARGETS = {
    'MIRAGE Top-32': (85.63, 85.47, 85.79, 92.60),
    'MIRAGE Top-10': (87.30, 88.32, 85.43, 93.67),
    'MMLU Top-10':   (90.06, 76.23, 78.29, 92.88),
}
print(f'{"실험":<20} {"Acc":>8} {"Rec":>8} {"F1":>8} {"AUC":>8}')
print('-'*50)
for label, ds, pdir, ck in [
    ('MIRAGE Top-32','medrag',   'data/processed/mirage','medrag'),
    ('MIRAGE Top-10','medrag',   'data/processed/mirage','medrag'),
    ('MMLU Top-10',  'wikipedia','data/processed/mmlu',  'wikipedia'),
]:
    try:
        r = _eval(ds, pdir, ck); t = TARGETS[label]
        print(f'{label:<20} {r[0]:>7.2f}% {r[1]:>7.2f}% {r[2]:>7.2f}% {r[3]:>7.2f}%')
        print(f'  목표:              {t[0]:>7.2f}% {t[1]:>7.2f}% {t[2]:>7.2f}% {t[3]:>7.2f}%')
    except FileNotFoundError: print(f'{label}: 체크포인트 없음')

In [ ]:
# ── 7. Query Reduction 분석 ───────────────────────────────────────────────
import json, os, numpy as np
from src.config import INPUT_DIM, STATS_DIR, EMBEDDINGS_DIR, K_RETRIEVE, DATA_SOURCES
from src.data_source import DataSource
from src.rag_router import RAGRouter

emb_dir = f'{EMBEDDINGS_DIR}/mirage'
sources = [DataSource.from_files(c, 'medrag', f'{emb_dir}/{c}_index.faiss',
                                  f'{emb_dir}/{c}_chunks.json',
                                  f'{STATS_DIR}/{c}_stats.json')
           for c in DATA_SOURCES['medrag']]

router = RAGRouter.load('checkpoints/medrag_model_best.pth',
                        'checkpoints/medrag_scaler.pkl',
                        sources, 'medrag', threshold=0.5)

with open('data/processed/mirage/train_test_split.json') as f:
    split_dict = json.load(f)

n_queries = n_selected = 0
source_counts = {s.source_id: 0 for s in sources}

for benchmark in ['pubmedqa', 'medqa', 'bioasq', 'medmcqa', 'mmlu-med']:
    emb_path = f'{emb_dir}/{benchmark}_query_embeddings.npy'
    ids_path = f'{emb_dir}/{benchmark}_query_ids.json'
    if not os.path.exists(emb_path): continue
    q_vecs = np.load(emb_path).astype(np.float32)
    with open(ids_path) as f: q_ids = json.load(f)
    for qid, qvec in zip(q_ids, q_vecs):
        if split_dict.get(qid) != 'test': continue
        selected = router.route(qvec)
        n_queries += 1; n_selected += len(selected)
        for s in selected: source_counts[s.source_id] += 1

n_src = len(sources)
reduction = (1 - n_selected / (n_queries * n_src)) * 100
print(f'=== medrag Query Reduction (test) ===')
print(f'  총 test queries: {n_queries}')
print(f'  평균 선택 source 수: {n_selected/n_queries:.2f} / {n_src}')
print(f'  query reduction: {reduction:.1f}%  (논문: 28~71%)')
for sid, cnt in source_counts.items():
    print(f'  {sid}: {cnt} ({cnt/n_queries*100:.1f}%)')


In [ ]:
import subprocess, time, requests

!pip install -q vllm openai

!pkill -f "vllm" 2>/dev/null || true
time.sleep(2)

vllm_log = open('/content/vllm.log', 'w')
vllm_proc = subprocess.Popen(
    ['python3', '-m', 'vllm.entrypoints.openai.api_server',
     '--model', 'unsloth/Meta-Llama-3.1-8B-Instruct',
     '--port', '8000', '--max-model-len', '32768',
     '--gpu-memory-utilization', '0.90', '--dtype', 'bfloat16'],
    stdout=vllm_log, stderr=vllm_log)
print(f'vLLM PID: {vllm_proc.pid} — 모델 로딩 3~5분 소요')

for i in range(360):
    time.sleep(5)
    if vllm_proc.poll() is not None:
        !tail -20 /content/vllm.log
        raise RuntimeError('vLLM 시작 실패')
    try:
        if requests.get('http://localhost:8000/health', timeout=2).status_code == 200:
            print(f'✅ vLLM 준비 ({i*5+5}초)'); break
    except Exception: pass
else:
    raise RuntimeError('vLLM 시간 초과')

!python scripts/06_evaluate.py --config experiments/mirage_top32.yaml --mode no_rag
!python scripts/06_evaluate.py --config experiments/mirage_top32.yaml --mode rag_all
!python scripts/06_evaluate.py --config experiments/mirage_top32.yaml --mode ragroute

!python scripts/06_evaluate.py --config experiments/mmlu_top10.yaml --mode no_rag
!python scripts/06_evaluate.py --config experiments/mmlu_top10.yaml --mode rag_all
!python scripts/06_evaluate.py --config experiments/mmlu_top10.yaml --mode ragroute

In [ ]:
import json, os

files = sorted(f for f in os.listdir('results') if f.endswith('.json')) if os.path.isdir('results') else []
if files:
    print(f'{"파일":<40} {"Accuracy":>10} {"N":>6}')
    print('-'*58)
    for fn in files:
        with open(f'results/{fn}') as f: r = json.load(f)
        print(f'{fn:<40} {r.get("accuracy",float("nan")):>9.2f}% {r.get("n_total","?"):>6}')
else:
    print('결과 없음 — Cell 9 실행 후 확인')

print()
print('논문 목표 (MIRAGE Top-32): No-RAG 67.04% / RAG-all 72.22% / RAGRoute 72.24%')
print('논문 목표 (MMLU Top-10):   No-RAG 66.67% / RAG-all 68.18% / RAGRoute 70.45%')

In [ ]:
import os, subprocess
from IPython.display import Image, display

os.makedirs('figures', exist_ok=True)
r = subprocess.run(['python', 'scripts/07_plot_results.py'], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[:300])

for fn in ['fig1_mirage_accuracy.png','fig2_mmlu_accuracy.png',
           'fig3_classification_metrics.png','fig4_query_reduction.png']:
    if os.path.exists(f'figures/{fn}'): display(Image(f'figures/{fn}'))

In [ ]:
# -- 11. 단위 테스트 ------------------------------------------------------
# 전체 테스트 출력이 길어도 실패 원인을 숨기지 않도록 tail을 쓰지 않습니다.
!python -m pytest tests/ -v --tb=short


In [ ]:
import os, sys, shutil, subprocess
from google.colab import drive, userdata

REPO_PATH  = '/content/ragroute_recreation'
DRIVE_ROOT = '/content/drive/MyDrive/ragroute'

drive.mount('/content/drive')
os.chdir(REPO_PATH); sys.path.insert(0, REPO_PATH)

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

def _link(local, sub):
    d = f'{DRIVE_ROOT}/{sub}'; os.makedirs(d, exist_ok=True)
    if os.path.islink(local): os.remove(local)
    elif os.path.isdir(local): shutil.rmtree(local)
    os.symlink(d, local)

for local, sub in [('data/embeddings','embeddings'), ('data/processed','processed'),
                   ('data/stats','stats'), ('data/raw','raw'),
                   ('checkpoints','checkpoints'), ('results','results')]:
    _link(local, sub)

for c in ['textbooks', 'statpearls']:
    d = f'data/raw/mirage/{c}'
    ok = os.path.isdir(d) and bool(os.listdir(d))
    print(f'  {c}: {"✅" if ok else "❌ Cell 2 재실행 필요"}')

print('\n복구 완료')